<img src="https://github.com/naserhusovic-droid/DSB-Unit-02/blob/main/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Descriptive statistics and data joins in `pandas`

---

### About
A guided walkthrough of calculating descriptive statistics with `pandas`, summarising data using `groupby` and joining data.

### Learning Objectives
- Define different types of data.
- Explore descriptive statistics of a data set using `pandas`.
- Produce summary statistics with respect to some categorical variable using `groupby` in `pandas`.
- Explain different kinds of join and merge data sets together using `pandas`.


### Notebook Guide
- Scenario
- Calculating descriptive statistics in `pandas`
    - Re-cap: data types
- Group-level analysis in `pandas` using **split-apply-combine** (`groupby`)
- Merging data sets together with `pandas`

# Scenario

You're an analyst working for CoolBricks - a new toy company who wants to compete with LEGO by producing their own bricks and sets. To better understand what kind of sets to create and at what price points, you are tasked with analyzing LEGO's catalogue over time.

The data comes from Kaggle: [https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time](https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time)

We are going to use `pandas` to read and explore this data.

The question we want to answer is: **How do LEGO products vary across different types/themes?**

# Calculating descriptive statistics in `pandas`
---

In [4]:
import pandas as pd

Let's read in the data and preview it.

In [5]:
lego_df = pd.read_csv("https://raw.githubusercontent.com/naserhusovic-droid/DSB-Unit-02/refs/heads/main/data/lego.csv")
lego_df.head()

,Set_ID,Name,Year,Theme,Theme_Group,Subtheme,Category,Packaging,Num_Instructions,Availability,Pieces,Minifigures,Owned,Rating,USD_MSRP,Total_Quantity,Current_Price
0,75-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},16.0,NaN,10.0,0.0,NaN,NaN,NaN
1,77-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},20.0,NaN,11.0,0.0,NaN,NaN,NaN
2,077-1,Pre-School Set,1975,Duplo,Pre-school,NaN,Normal,{Not specified},0,{Not specified},21.0,NaN,10.0,0.0,NaN,0.0,NaN
3,78-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},32.0,NaN,8.0,0.0,NaN,NaN,NaN
4,78-3,Basic Set,1975,Samsonite,Vintage,Basic set,Normal,Box,0,{Not specified},330.0,NaN,10.0,0.0,NaN,0.0,NaN


Let's look at the column information

In [6]:
lego_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14936 entries, 0 to 14935
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Set_ID            14936 non-null  object 
 1   Name              14936 non-null  object 
 2   Year              14936 non-null  int64  
 3   Theme             14936 non-null  object 
 4   Theme_Group       14915 non-null  object 
 5   Subtheme          11495 non-null  object 
 6   Category          14936 non-null  object 
 7   Packaging         14936 non-null  object 
 8   Num_Instructions  14936 non-null  int64  
 9   Availability      14936 non-null  object 
 10  Pieces            13133 non-null  float64
 11  Minifigures       7686 non-null   float64
 12  Owned             14771 non-null  float64
 13  Rating            14936 non-null  float64
 14  USD_MSRP          5837 non-null   float64
 15  Total_Quantity    12276 non-null  float64
 16  Current_Price     5442 non-null   float6

---

## Recap: data types

We can categorize data into types:

- **Categorical** (discrete categories)
  - Nominal (no inherent ordering, e.g. color)
  - Ordinal (discrete, ordered categories where there is an inherent order e.g., t-shirt sizes S/M/L, or number of visits to the dentist)
- **Continuous** (numeric values along a scale)
  - Interval (zero doesn't mean an absence, e.g. temperature)
  - Ratio (zero means an absence of the value, e.g. height)

## Give an example from the LEGO data of:

- A **nominal** type - see below
- An **ordinal** type - see below
- An **interval** type - see below
- A **ratio** type - see below

Let's get some high-level information about prices

### Nominal
- Set_ID
- Theme
- Theme_Group
- Subtheme
- Name
- Category
- Packaging
- Availability

### Ordinal
- Rating

### Interval
- USD_MSRP
- Current_Price

### Ratio
- Pieces
- Minifigures
- Owned
- Total_Quantity
- Year

In [7]:
lego_df["USD_MSRP"].mean()

np.float64(39.96704985437725)

In [8]:
lego_df["USD_MSRP"].median()

19.99

We can also perform multiple aggregations at once

In [9]:
lego_df["USD_MSRP"].agg(["min", "max"])

,USD_MSRP
min,1.49
max,849.99


Or use `describe` to get an overview of *all* numeric data

In [10]:
lego_df.describe()

,Year,Num_Instructions,Pieces,Minifigures,Owned,Rating,USD_MSRP,Total_Quantity,Current_Price
count,14936.000000,14936.000000,13133.000000,7686.000000,14771.000000,14936.000000,5837.000000,12276.000000,5442.000000
mean,2006.750736,1.196036,237.935734,2.709862,2045.236951,1.826627,39.967050,4.708618,90.164663
std,11.524612,2.087763,473.090631,2.955427,2972.330218,1.950909,55.006213,20.995961,193.452605
min,1975.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.490000,0.000000,0.050000
25%,2000.000000,0.000000,25.000000,1.000000,90.000000,0.000000,9.990000,0.000000,15.000000
50%,2009.000000,0.000000,81.000000,2.000000,744.000000,0.000000,19.990000,0.000000,36.000000
75%,2016.000000,2.000000,262.000000,3.000000,2828.500000,3.900000,49.990000,3.000000,92.000000
max,2023.000000,48.000000,11695.000000,80.000000,29429.000000,4.900000,849.990000,1063.000000,5499.990000


In our research question, we want to know about different themes.

The `Theme` and `Theme_Group` columns look interesting to investigate.

How many different themes are there and what are they?

In [14]:
themed_lego = lego_df["Theme"].unique()
themed_lego

array(['PreSchool', 'Duplo', 'Samsonite', 'Trains',
       'Building Set with People', 'Books', 'LEGOLAND', 'Hobby Set',
       'Gear', 'Universal Building Set', 'Basic', 'Dacta', 'Minitalia',
       'Homemaker', 'Technic', 'Service Packs', 'Promotional', 'Town',
       'Castle', 'Space', 'Fabuland', 'Scala', 'Education', 'Boats',
       'Model Team', 'Assorted', 'Pirates', 'Belville', 'Creator',
       'Freestyle', 'Primo', 'Aquazone', 'Time Cruisers', 'Western',
       'Classic', 'Adventurers', 'Znap', 'Racers', 'Mindstorms',
       'Seasonal', 'Rock Raiders', 'Star Wars', 'Studios', 'Baby',
       'Action Wheelers', 'Sports', 'Bulk Bricks', 'Advanced models',
       'Mickey Mouse', 'Bionicle', 'Jack Stone', 'Harry Potter',
       'Dinosaurs', 'Alpha Team', 'Explore', 'Spybotics', 'Galidor',
       'Miscellaneous', 'Island Xtreme Stunts', 'Clikits', 'World City',
       '4 Juniors', 'Spider-Man', 'Discovery', 'Quatro',
       'Make and Create', 'City', 'Factory', 'Vikings', 'Dino 201

In [16]:
lego_df["Theme_Group"].unique()

array(['Pre-school', 'Vintage', 'Modern day', 'Miscellaneous', 'Basic',
       'Educational', 'Technical', 'Historical', 'Action/Adventure',
       'Junior', 'Girls', 'Model making', 'Racing', 'Licensed',
       'Constraction', nan, 'Art and crafts'], dtype=object)

In [17]:
lego_df["Theme_Group"].value_counts()

,count
Theme_Group,
Miscellaneous,3407
Modern day,2228
Licensed,2068
Pre-school,1547
Action/Adventure,1177
Basic,827
Model making,725
Technical,594
Constraction,491


In [18]:
lego_df["Theme"].value_counts()

,count
Theme,
Duplo,1278
Gear,1232
Collectable Minifigures,804
City,770
Star Wars,723
...,...
LEGO Games,2
Stranger Things,1
Boost,1


It looks like themes are contained in theme groups (of which there are fewer).

How many products in each theme group?

In [21]:
lego_df.groupby(["Theme"])

This shows us the **mode** of `Theme_Group`.  This is the time that mode is used: for looking at the most common category in categorical data.

Now that we have a sense of what themes exist, we want to know **how prices *vary* across them.**

To do this, we can analyze each group separately using `groupby`, which is a method that performs a "split-apply-combine".

# Group-level analysis in `pandas` using **split-apply-combine** (`groupby`)
---

If we want summary statistics _with respect to some categorical variable_, for example the price of a LEGO set across theme groups, we can use the **Split-Apply-Combine** technique. (This is sometimes called **MapReduce**, but is more of a special case of MapReduce).

* **Split**: Separate your data into different DataFrames, one for each category.
* **Apply**: On each split-up DataFrame, apply some function or transformation (for example, the mean).
* **Combine**: Take the results and combine the split-up DataFrames back into one aggregate DataFrame.

    
<img src="https://github.com/naserhusovic-droid/DSB-Unit-02/blob/main/assets/split_apply_combine.png?raw=1" style="width: 500px;">

_**Source**: Python for Data Analysis, Wes McKinney_

This might sound complicated, but it's actually only two commands in `pandas` (the **Combine** step is done for us).

You can check out a more detailed visual [here](../assets/Split-Apply-Combine-Visual.pdf)!

To answer the question **"What is the average prices of a LEGO set across themes?"** we need to first define the groups.

This is the **Split** step.

This gives us a `groupby` object on which we need to *apply* some summarization.

When we define the column(s) to summarize (e.g. price) and how to summarize them (e.g. taking the median), `pandas` performs *both* the apply and combine steps.

Finally, we can sort these values in descending order, making this our full command:

# Merging data sets together with `pandas`
---
If we wanted to understand consumer sentiment at the point when LEGO sets were released, we could join some CPI (Consumer Price Index), such as from the table here: https://www.usinflationcalculator.com/inflation/consumer-price-index-and-annual-percent-changes-from-1913-to-2008/

Let's read in a download of this CPI  data that happens to be in JSON (JavaScript Object Notation) format and not a CSV.

In [ ]:
cpi = pd.read_json("../data/cpi.json")
cpi.head()

Every data set you read in to `pandas` becomes a `DataFrame` regardless of its original source.

Even though this data wasn't from a CSV, but a JSON file, it's still a `DataFrame`.

Let's join this data to our LEGO DataFrame.

For this, we can use the `pandas` `merge` method

Oops, what happened? We lost some rows!

Ah, our CPI data doesn't start until 1980 but LEGO sets start in 1975!

If we change the merge to a LEFT JOIN, we will keep *all* records in `lego` regardless of whether there's a match in `cpi`

Now we have the right number of records and we can see missing data where there is no CPI information available

# Conclusions and Takeaways

- `pandas` lets us calculate descriptive statistics on individual columns, or multiple columns at once
- `pandas` lets us calculate individual descriptive statistics or multiple aggregations at once
- The `describe` method gives us a high-level numeric summary of our data
- To do statistical analysis on subgroups, we can use `groupby`
- We can join and merge data together from different sources into `pandas` `DataFrames`